![SAAI](https://i.imgur.com/YHRhgqv.png)

**English** | [العربية](data_exploration_and_preparation.ar.ipynb)

# FURSAH: Prepare and understand announcement data

**Our goal:** turn raw announcements into a table we can inspect and save, while keeping the original evidence. We will follow the same ten supplied fictional announcements, then reuse the workflow on all 1,000.

**Our route:** read → explore → clean → extract → inspect unknowns → review related sources → review deadlines → process all sources → save.

There are **nine small exercises**, each requiring 1–3 expressions. Before every exercise you will see what the operation does, why it matters and a working example. Replace `...` before running that cell. Run cells in order.

The supplied [preparation_helpers.py](preparation_helpers.py) handles repetitive extraction, date processing and saving. You do not need to edit it to finish the lesson. Its code is available to inspect; we explain its inputs, outputs and limitations here. Your task is to use the workflow thoughtfully and inspect its results—not implement every mechanism from scratch.

**Result:** one prepared row per source, including selected facts, original text, unknown values and review flags. This is a preparation foundation, not a complete opportunity database or a recommendation backend.

## Setup — run the supplied cells

Use Python 3.11+ in Jupyter. Open the notebook from the project's `data/` folder and keep `preparation_helpers.py` beside it. If imports or paths fail, check `Path.cwd()` and the selected kernel.

**pandas** provides a `DataFrame`: a table with rows and columns. `display()` shows the table. `Path` handles file paths. These are enough for this lesson; no model download or API key is needed.

If packages are missing, uncomment `%pip` below. It installs into this notebook's environment; `clear_output()` clears the installation log. Restart the kernel if requested.

In [ ]:
# Uncomment if needed / أزل التعليق عند الحاجة
# %pip install pandas ipython
from IPython.display import clear_output
clear_output(wait=False)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from preparation_helpers import (
    extract_facts, related_sources, review_deadlines,
    prepare_announcements, save_prepared_data,
)
DATA_PATH = Path("raw/raw_opportunities.csv")

## 1. Read an announcement

The CSV contains **1,000 fictional Arabic announcements**: emails, social posts, brochure text, group messages and noticeboards. A row is an announcement, not necessarily a unique opportunity. Expect reposts, updates, missing facts and disagreements.

| Column | Meaning |
|---|---|
| `source_id` | Source identifier; preserve it |
| `source_type` | Announcement format |
| `source_name` | Publisher |
| `published_at` | Publication time—not a deadline or verification time |
| `content` | Original announcement text |

`pd.read_csv()` reads CSV safely, including commas and newlines inside quoted fields. `dtype="string"` keeps these fields as text; `utf-8-sig` reads the file encoding. `.head(3)` previews three rows.

**Two ideas before starting:** an announcement is one source row; an opportunity is the activity that several sources may describe. Preparing 1,000 announcements does not mean finding 1,000 unique opportunities. We preserve sources first; your team later decides how to link them to opportunities.

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig", dtype="string")
print("Sources:", len(raw))
display(raw.head(3))

`raw.iloc[0]` selects the first row. `row["content"]` reads its text. Read one complete announcement before choosing what to extract.

In [ ]:
row = raw.iloc[0]
print(row["source_id"])
print(row["content"])

### Your turn · Read the next source

Choose the second row with `.iloc[1]` and print its content. **Two expressions.** Expect `SRC-0002`, an email mentioning 50 SAR and 4 October 2026.

In [ ]:
second = ...
print(...)

## 2. Explore, then work on a small sample

**Exploration** helps us notice different formats before processing everything. Selecting `raw["source_type"]` gives one column; `.value_counts()` counts its categories. `.isna().sum()` counts missing CSV cells. A nonempty text can still omit the fee or deadline.

`.head(5).copy()` makes an independent five-row sample. We use a copy so the original table remains unchanged.

In [ ]:
display(raw["source_type"].value_counts())
display(raw.isna().sum())
example_sample = raw.head(5).copy()
print("Example sample:", len(example_sample))

### Your turn · Choose ten sources

Follow the sample example, changing 5 to 10. **One expression.** Expect ten rows; these include different formats and a related-source pair.

In [ ]:
sample = ...
display(sample[["source_id", "source_type"]])

The double brackets above select two columns for display. We will keep using `sample` through the following steps, so each section builds on the previous one.

## 3. Clean formatting, keep the meaning

Cleaning fixes formatting—not facts. Keep numbers, dates, line breaks and negation such as “لا”. `.str.strip()` removes only surrounding whitespace from every text in a column. Save the result in a new column, `cleaned_content`, leaving `content` intact.

Here is a complete example. Unchanged rows are valid; do not add cleaning without an observed problem.

In [ ]:
example_text = pd.Series(["  لا تتطلب خبرة. الرسوم 100 ريال.  "])
display(example_text.str.strip())

### Your turn · Clean the sample

Apply the demonstrated `.str.strip()` to `sample["content"]`. **One expression.** Compare the two columns; words and amounts must stay unchanged.

In [ ]:
sample["cleaned_content"] = ...
display(sample[["content", "cleaned_content"]].head(2))

## 4. Extract facts and inspect the evidence

**Extraction** turns wording into fields; **normalization** represents a known fact consistently, such as “المشاركة مجانية” → `0`. Evidence is the source text supporting the value.

The supplied `extract_facts(table)` takes cleaned source rows and returns a copy with these fields:

| Field | Meaning |
|---|---|
| `title_candidates` | All phrases in Arabic quotation marks |
| `title_suggestion` | A suggestion when exactly one phrase was found |
| `title_needs_review` | No single clear candidate; inspect the source |
| `fee_evidence` | Matched fee phrase, preserved for inspection |
| `cost_sar` | Numeric amount, explicit free `0`, or unknown |
| `activity_url` | Recognized activity-specific link |

**Limitations:** the helper recognizes a few observed wording patterns, not arbitrary Arabic. It may miss facts. Multiple quoted phrases may include the publisher as well as the title. Negation, multiple fee claims or updates require reading the source; a first match is not guaranteed truth. It never confirms eligibility or merges announcements.

You can inspect [the helper implementation](preparation_helpers.py), but you do not need to write its text patterns. First see it work on one source.

**Watch substring matches:** `تحدي` occurs inside `تحديد`; that does not make the activity a competition. Review varied contexts before extending rules. The supplied helpers are limited examples, not a comprehensive parser.

In [ ]:
one_prepared = extract_facts(sample.head(1))
display(one_prepared[["source_id", "title_suggestion", "fee_evidence", "cost_sar"]])

### Your turn · Prepare the ten sources

Call `extract_facts()` with `sample`, following the example. **One expression.** Expect ten rows, with source 1 free and source 2 costing 50 SAR.

In [ ]:
sample_facts = ...
display(sample_facts[["source_id", "title_suggestion", "fee_evidence", "cost_sar"]])

## 5. Unknown is different from free

Pandas shows unknown values as `<NA>` or `NaN`; JSON saves them as `null`. **Do not replace missing fees with zero.** Source 4 says its fee is unspecified, so it must stay unknown.

A **filter** selects matching rows. `.isna()` marks missing values; `.loc[condition, columns]` displays only those rows and columns. This complete example inspects title uncertainty. The boolean `title_needs_review` is already supplied by the helper.

In [ ]:
needs_title_review = sample_facts["title_needs_review"]
display(sample_facts.loc[needs_title_review, ["source_id", "title_candidates"]])

### Your turn · Find unknown fees

Use `.isna()` on `sample_facts["cost_sar"]` to complete the filter. **One expression.** Expect source 4 among the results. Read its text to explain why unknown is correct.

In [ ]:
missing_fee = ...
display(sample_facts.loc[missing_fee, ["source_id", "content", "cost_sar"]])

**Keep these meanings separate:** “يشترط إتقان SQL” is mandatory; “يفضل معرفة Excel” is preferred. Missing requirements do not mean no requirements. Our supplied example does not extract these fields; your team decides how to extend preparation for its application.

## 6. Related announcements need review

A **repost** repeats an announcement. An **explicit update** says which fact changed. A **contradiction** presents incompatible claims without a justified resolution.

`related_sources(table)` returns source rows sharing a nonmissing activity URL. It does **not** merge, delete or choose a correct claim. Missing links are not treated as a common identity. Compare organizer, dates and wording before deciding whether sources describe the same activity.

For example, on the first five rows, the following view is empty. An empty review view is a valid result.

A shared URL is a review signal, not proof of identity. If publishers differ, do not automatically treat the latest text as an authorized update. Retain both sources until you review their relationship and evidence.

In [ ]:
display(related_sources(sample_facts.head(5)))

### Your turn · Find the related pair

Call the same helper on all of `sample_facts`. **One expression.** Expect `SRC-0006` and `SRC-0007`, both quoting 120 SAR. Read the two texts: source 7 explicitly says “إعادة نشر”. Do not delete either source.

In [ ]:
review_pair = ...
display(review_pair[["source_id", "source_name", "cost_sar", "content"]])

**Decision checkpoint:** source 7 omits some facts stated in source 6. An omission does not cancel the earlier evidence. If another source claimed 150 SAR, keep both claims pending review. A verified deadline update changes that field only; it does not replace unrelated facts. Similarity alone is not permission to delete a source.

**Illustration, not a dataset answer:** the original activity runs 20–22 October. An update says “The start moved to 25 October” without naming an end. The new start is 25 October; the new end and duration are unknown. Do not guess that the end moved to 27 October. No extra code is required here; explain why to a teammate.

## 7. Review deadlines at an explicit time

**Evaluation time** is the “now” used for a decision. Use **18 October 2026 at 12:00 Asia/Riyadh**, matching the backend classroom setting. `pd.Timestamp()` represents this instant; `+03:00` is its timezone offset. Keep dataset dates unchanged.

`review_deadlines(table, evaluation_time)` reads the observed “آخر يوم للتقديم …” wording, retains the extracted date, and adds `passed`, `not_passed` or `uncertain` plus the evaluation time. It does not use publication time as a deadline, resolve updates or infer eligibility.

The source dates lack a cutoff hour/timezone. The supplied review policy is deliberately conservative: **before 17 October → passed; after 19 October → not passed; 17–19 October, missing or invalid → uncertain**. The extra review buffer avoids assuming an unstated timezone. This is not an exact-cutoff parser; an explicitly known cutoff would expire at equality. The helper applies the equivalent buffer when evaluation time changes.

A future deadline alone does not prove registration is open. We retain all source records, including expired ones.

**Source status is not current status:** if an announcement says registration has not opened and names a date that has passed, do not infer that it is now open. Retain the evidence and explain that current status needs verification. No extra coding exercise is required here.

In [ ]:
EVALUATION_TIME = pd.Timestamp("2026-10-18T12:00:00+03:00")
sample_prepared = review_deadlines(sample_facts, EVALUATION_TIME)
display(sample_prepared[["source_id", "deadline_date", "deadline_state"]])

As in section 5, filter a copy for inspection. `!=` means “not equal”. The example counts the states; your exercise creates a candidate view without removing rows from `sample_prepared`.

In [ ]:
display(sample_prepared["deadline_state"].value_counts())

### Your turn · Keep a separate candidate view

Complete the condition with `sample_prepared["deadline_state"] != "passed"`. **One expression.** Source 2 should be excluded from this view; source 1 stays uncertain, not confirmed available. The original ten rows remain.

In [ ]:
candidates = sample_prepared.loc[...].copy()
display(candidates[["source_id", "deadline_state"]])

## 8. Reuse the workflow on all sources

We have followed cleaning → extraction → deadline review. `prepare_announcements(table, evaluation_time)` runs **these same three steps** in that order. It returns one row per input source, preserves original text and keeps unknowns visible. It does not merge records, resolve contradictions, generate embeddings or add unmentioned facts.

This is a supplied convenience function, not a new algorithm to learn. First run it on 50 sources and inspect a few results. Then run the same call on the full table. No extra loop is needed.

In [ ]:
prepared_fifty = prepare_announcements(raw.head(50), EVALUATION_TIME)
print("Prepared sources:", len(prepared_fifty))
display(prepared_fifty[["source_id", "cost_sar", "deadline_state"]].head(3))

### Your turn · Prepare all 1,000 sources

Repeat the example with `raw` instead of `raw.head(50)`. **One expression.** Expect 1,000 rows; unknown fees and uncertain dates remain visible.

In [ ]:
prepared = ...
print("Prepared sources:", len(prepared))
display(prepared["deadline_state"].value_counts())

**Check preservation:** `.nunique()` counts distinct IDs. `.equals()` compares columns including their order. These supplied checks should show `1000`, `True`, `True`. They check source preservation, not extraction accuracy; still inspect original evidence for ambiguous and missing facts.

In [ ]:
print("Distinct IDs:", prepared["source_id"].nunique())
print("IDs preserved:", prepared["source_id"].equals(raw["source_id"]))
print("Original text preserved:", prepared["content"].equals(raw["content"]))

## 9. Save, reload, and explain your decisions

**JSON** stores records in a file. Unknown values become `null`, and Arabic stays readable. `save_prepared_data(table, path)` creates the destination folder, writes the JSON and returns its path. Re-running replaces the chosen JSON output. It does not change the raw CSV.

The example saves only the ten-source sample to `prepared/lesson_sample.json`. Your exercise uses a different path for the full result. These teaching exports do not overwrite the application's opportunity store. Stored calendar dates are not evidence of midnight cutoffs.

**What a row should look like:** it retains `source_id` and `content`, with observations such as `fee_evidence` and `cost_sar`. If the helper cannot establish a fee, it stays unknown. This is a source record for review, not a finished API response.

In [ ]:
sample_path = save_prepared_data(sample_prepared, Path("prepared/lesson_sample.json"))
print(sample_path)

### Your turn · Save the full prepared table

Call `save_prepared_data()` with `prepared` and the supplied `output_path`, following the sample. **One expression.** Expect a JSON file containing all 1,000 source records.

In [ ]:
output_path = Path("prepared/source_preparation.json")
saved_path = ...
print(saved_path)

Finally, `pd.read_json()` reloads the file. `orient="records"` means a list of row records; `convert_dates=False` keeps stored dates as text for inspection. `.tolist()` makes ordinary Python lists for comparing saved and original values. Expect 1,000 rows and both comparisons `True`.

In [ ]:
reloaded = pd.read_json(saved_path, orient="records", convert_dates=False)
print("Reloaded sources:", len(reloaded))
print("IDs preserved:", reloaded["source_id"].tolist() == raw["source_id"].tolist())
print("Text preserved:", reloaded["content"].tolist() == raw["content"].tolist())
display(reloaded[["source_id", "cost_sar", "deadline_state"]].head(3))

## What you have achieved

You loaded, inspected, cleaned, prepared and saved all source rows using a supplied workflow. You checked unknown fees, ambiguous titles, related sources and expired or uncertain deadlines. You did not need to write the supporting extraction patterns or export machinery.

**Explain three decisions:** why missing fees are not zero; why shared links do not authorize deletion; and why a future deadline does not confirm eligibility. These explanations matter as much as filling the code blanks.

**Next for your team:** extend the preparation for your chosen application fields, review updates and conflicts, decide how to represent opportunities, and connect your own storage through the [backend guide](../backend/README.md). The teaching table is not the fixed API response schema. No framework or storage choice is compulsory.

Embeddings are optional later: they represent text numerically for similarity search, but cannot prove that announcements are identical. They are not required for this lesson.

[Data guide](README.md) · [Supplied helpers](preparation_helpers.py)

**First step after the notebook:** open your exported data, choose an activity supported by actual sources, then read it from your store in `repository.py` and map it to `OpportunityDetailsResponse`. Try its ID in `/docs` before expanding discovery. The API does not read cell outputs or execute this notebook automatically. Follow the [integration steps](../backend/README.md#first-working-integration).

Before implementing the backend, complete the [field-mapping worksheet](README.md#mapping) for one activity. Identify what is needed for the first details response and what can remain unknown. Completing these exercises does not complete the search engine; this is the next integration milestone.

![SAAI](https://i.imgur.com/beBR1m2.png)